# SD Scripts Colab
Created by [licyk](https://github.com/licyk)

Jupyter Notebook 仓库：[licyk/sd-webui-all-in-one](https://github.com/licyk/sd-webui-all-in-one)


## 简介
一个在 [Colab](https://colab.research.google.com/) 部署 [sd-scripts](https://github.com/kohya-ss/sd-scripts) 的 Jupyter Notebook，可用于 Stable Diffusion 模型的训练。

这个 Colab 脚本只是写来玩的，还有用来开发和测试管理模块的功能。如果要用这个脚本进行训练就参考 [SD Scripts Kaggle Jupyter NoteBook](https://github.com/licyk/sd-webui-all-in-one?tab=readme-ov-file#sd-scripts-kaggle-jupyter-notebook)，毕竟同款核心。

In [ ]:
# @title 👇 参数配置
# SD WebUI All In One 功能初始化部分, 通常不需要修改
# 如果需要查看完整代码实现, 可阅读: https://github.com/licyk/sd-webui-all-in-one/blob/main/sd_webui_all_in_one
#################################################################################################################
import os
import sys
import shutil
from pathlib import Path
try:
    _ = JUPYTER_ROOT_PATH  # type: ignore # noqa: F821
except Exception:
    JUPYTER_ROOT_PATH = os.getcwd()
try:
    _ = INIT_SD_WEBUI_ALL_IN_ONE_CORE  # type: ignore # noqa: F821
    print("[Core] SD WebUI All In One 核心模块已初始化")
except Exception:
    print("[Core] 初始化 SD WebUI All In One 核心模块中")
    !python -m pip install sd-webui-all-in-one --upgrade
    from sd_webui_all_in_one import logger, VERSION, SDScriptsManager
    INIT_SD_WEBUI_ALL_IN_ONE_CORE = True
    logger.info("SD WebUI All In One 核心模块初始化完成, 版本: %s", VERSION)
# @markdown ---
##############################################################################

# @markdown ## 环境设置
# @markdown - 工作路径, 通常不需要修改
WORKSPACE = "/content" # @param {"type":"string","placeholder":"填写工作路径"}
# @markdown - 工作路径中文件夹名称, 通常不需要修改
WORKFOLDER = "sd-scripts" # @param {"type":"string","placeholder":"填写工作文件夹"}
# @markdown - sd-scripts 仓库地址
SD_SCRIPTS_REPO = "https://github.com/kohya-ss/sd-scripts" # @param {"type":"string","placeholder":"填写 sd-scripts 仓库的 Git 地址"}
# @markdown - sd-scripts 依赖文件名
SD_SCRIPTS_REQUIREMENTS = "requirements.txt" # @param {"type":"string","placeholder":"填写 sd-scripts 依赖记录的文件名"}
# @markdown - PyTorch 版本
TORCH_VER = "torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124" # @param {"type":"string","placeholder":"填写 PyTorch 软件包名和版本"}
# @markdown - xFormers 版本
XFORMERS_VER = "xformers==0.0.28.post3" # @param {"type":"string","placeholder":"填写 xFormers 软件包名和版本"}
# @markdown - 使用 uv 加速 Python 软件包安装, 修改为 True 为启用, False 为禁用
USE_UV = True # @param {type:"boolean"}
# @markdown - PyPI 主镜像源
PIP_INDEX_MIRROR = "https://pypi.python.org/simple" # @param {"type":"string","placeholder":"填写 PyPI 镜像地址"}
# @markdown - PyPI 扩展镜像源
PIP_EXTRA_INDEX_MIRROR = "https://download.pytorch.org/whl/cu124" # @param {"type":"string","placeholder":"填写 PyPI 镜像地址"}
# @markdown - PyPI 额外镜像源
PIP_FIND_LINKS_MIRROR = "" #@param {type:"string"}
# @markdown - 用于下载 PyTorch 的镜像源
PYTORCH_MIRROR = "https://download.pytorch.org/whl/cu124" # @param {"type":"string","placeholder":"填写 PyTorch 镜像地址"}
# PyPI 扩展镜像源
PIP_FIND_LINKS_MIRROR = "https://download.pytorch.org/whl/cu121/torch_stable.html"
HUGGINGFACE_MIRROR = "https://hf-mirror.com" # HuggingFace 镜像源
GITHUB_MIRROR = [ # Github 镜像源
    "https://ghfast.top/https://github.com",
    "https://mirror.ghproxy.com/https://github.com",
    "https://ghproxy.net/https://github.com",
    "https://gh.api.99988866.xyz/https://github.com",
    "https://gh-proxy.com/https://github.com",
    "https://ghps.cc/https://github.com",
    "https://gh.idayer.com/https://github.com",
    "https://ghproxy.1888866.xyz/github.com",
    "https://slink.ltd/https://github.com",
    "https://github.boki.moe/github.com",
    "https://github.moeyy.xyz/https://github.com",
    "https://gh-proxy.net/https://github.com",
    "https://gh-proxy.ygxz.in/https://github.com",
    "https://wget.la/https://github.com",
    "https://kkgithub.com",
    "https://gitclone.com/github.com",
]
# @markdown - 检查可用的 GPU, 当 GPU 不可用时强制终止安装进程
CHECK_AVALIABLE_GPU = False # @param {type:"boolean"}
# @markdown - 重试下载次数
RETRY = 3 # @param {type:"slider", min:1, max:128, step:1}
# @markdown - 下载线程
DOWNLOAD_THREAD = 16 # @param {type:"slider", min:1, max:128, step:1}
# @markdown - 启用 TCMalloc 内存优化
ENABLE_TCMALLOC = True # @param {type:"boolean"}
#@markdown - 启用 CUDA Malloc 显存优化
ENABLE_CUDA_MALLOC = True #@param {type:"boolean"}
#@markdown - 更新内核
UPDATE_CORE = True #@param {type:"boolean"}

# @markdown ---
##############################################################################

# @markdown ## sd-scripts 版本设置
# @markdown - sd-scripts 分支, 可切换成 main / dev 或者其它分支, 留空则不进行切换
SD_SCRIPTS_BRANCH = "dev" # @param {"type":"string","placeholder":"填写 sd-scripts 分支名"}
# @markdown - 切换 sd-scripts 的版本到某个 Git 提交记录上, 留空则不进行切换
SD_SCRIPTS_COMMIT = "" # @param {"type":"string","placeholder":"填写 sd-scripts 版本提交哈希值"}

# @markdown ---
##############################################################################

# @markdown ## 模型上传设置, 使用 HuggingFace / ModelScope 上传训练好的模型
# @markdown HuggingFace: https://huggingface.co
# @markdown ModelScope: https://modelscope.cn
# @markdown - 使用 HuggingFace 保存训练好的模型
USE_HF_TO_SAVE_MODEL = False # @param {type:"boolean"}
# @markdown - 使用 ModelScope 保存训练好的模型
USE_MS_TO_SAVE_MODEL = False # @param {type:"boolean"}

# @markdown ## Token 配置, 用于上传 / 下载模型 (部分模型下载需要 Token 进行验证)
# @markdown HuggingFace Token 在 Account -> Settings -> Access Tokens 中获取
# @markdown - HuggingFace Token
HF_TOKEN = "" # @param {"type":"string","placeholder":"填写 HuggingFace Token"}
# @markdown ModelScope Token 在 首页 -> 访问令牌 -> SDK 令牌 中获取
# @markdown - ModelScope Token
MS_TOKEN = "" # @param {"type":"string","placeholder":"填写 ModelScope SDK 令牌"}

# @markdown ## 用于上传模型的 HuggingFace 模型仓库的 ID, 当仓库不存在时则尝试新建一个
# @markdown - HuggingFace 仓库的 ID (格式: "用户名/仓库名")
HF_REPO_ID = "" # @param {"type":"string","placeholder":"填写 HuggingFace 仓库 ID"}
# @markdown - HuggingFace 仓库的种类
HF_REPO_TYPE = "model" # @param ["model", "dataset", "space"]
# @markdown HuggingFace 仓库类型和对应名称:</br>
# @markdown model: 模型仓库</br>
# @markdown dataset: 数据集仓库</br>
# @markdown space: 在线运行空间仓库</br>

# @markdown ## 用于上传模型的 ModelScope 模型仓库的 ID, 当仓库不存在时则尝试新建一个
# @markdown - ModelScope 仓库的 ID (格式: "用户名/仓库名")
MS_REPO_ID = "" # @param {"type":"string","placeholder":"填写 ModelScope 仓库 ID"}
# @markdown - ModelScope 仓库的种类
MS_REPO_TYPE = "model" # @param ["model", "dataset", "space"]
# @markdown ModelScope 仓库类型和对应名称:</br>
# @markdown model: 模型仓库</br>
# @markdown dataset: 数据集仓库</br>
# @markdown space: 创空间仓库</br>

# @markdown ## 设置自动创建仓库时仓库的可见性, 通常保持默认即可
# @markdown - 设置新建的 HuggingFace 仓库可见性
HF_REPO_VISIBILITY = False # @param {type:"boolean"}
# @markdown - 设置新建的 ModelScope 仓库可见性
MS_REPO_VISIBILITY = False # @param {type:"boolean"}

# @markdown ## Git 信息设置, 可以使用默认值
# @markdown - Git 的邮箱
GIT_USER_EMAIL = "username@example.com" # @param {"type":"string","placeholder":"填写邮箱"}
# @markdown - Git 的用户名
GIT_USER_NAME = "username" # @param {"type":"string","placeholder":"填写用户名"}

# @markdown ---
##############################################################################

# @markdown ## 训练日志设置, 可使用 TensorBoard / WandB 记录训练日志, 使用 WandB 可远程查看实时训练日志
# @markdown 使用 WandB 需要填写 WANDB_TOKEN</br>
# @markdown 如果 TensorBoard 和 WandB 同时使用, 可以改成 all</br>
# @markdown - 使用的日志记录工具 (tensorboard / wandb / all)
LOG_MODULE = "tensorboard" # @param ["tensorboard", "wandb", "all"]

# @markdown ## WandB Token 设置
# @markdown WandB Token 可在 https://wandb.ai/authorize 中获取
# @markdown - WandB Token
WANDB_TOKEN = "" # @param {"type":"string","placeholder":"填写 WandB Token"}

# @markdown ---
##############################################################################

# 路径设置, 通常保持默认即可
# @markdown - 训练集保存的路径
INPUT_DATASET_PATH = "/content/dataset" # @param {"type":"string","placeholder":"填写训练集保存路径"}
# @markdown - 训练时模型保存的路径
OUTPUT_PATH = "/content/working/model" # @param {"type":"string","placeholder":"填写训练输出的模型保存路径"}
# @markdown - 模型下载到的路径
SD_MODEL_PATH = "/content/sd-models" # @param {"type":"string","placeholder":"填写下载模型的路径"}

# @markdown ---
##############################################################################

# @markdown ## 训练模型设置, 在安装时将会下载选择的模型
SD_MODEL = []

# @markdown - Stable Diffusion 模型
v1_5_pruned_emaonly = False # @param  {type:"boolean"}
animefull_final_pruned = False # @param  {type:"boolean"}
sd_xl_base_1_0_0_9vae = False # @param  {type:"boolean"}
sd_xl_refiner_1_0_0_9vae = False # @param  {type:"boolean"}
sd_xl_turbo_1_0_fp16 = False # @param  {type:"boolean"}
animagine_xl_3_0_base = False # @param  {type:"boolean"}
animagine_xl_3_0 = False # @param  {type:"boolean"}
animagine_xl_3_1 = False # @param  {type:"boolean"}
animagine_xl_4_0 = False # @param  {type:"boolean"}
animagine_xl_4_0_opt = False # @param  {type:"boolean"}
holodayo_xl_2_1 = False # @param  {type:"boolean"}
kivotos_xl_2_0 = False # @param  {type:"boolean"}
clandestine_xl_1_0 = False # @param  {type:"boolean"}
UrangDiffusion_1_1 = False # @param  {type:"boolean"}
RaeDiffusion_XL_v2 = False # @param  {type:"boolean"}
kohaku_xl_delta_rev1 = False # @param  {type:"boolean"}
kohakuXLEpsilon_rev1 = False # @param  {type:"boolean"}
kohaku_xl_epsilon_rev2 = False # @param  {type:"boolean"}
kohaku_xl_epsilon_rev3 = False # @param  {type:"boolean"}
kohaku_xl_zeta = False # @param  {type:"boolean"}
starryXLV52_v52 = False # @param  {type:"boolean"}
heartOfAppleXL_v20 = False # @param  {type:"boolean"}
heartOfAppleXL_v30 = False # @param  {type:"boolean"}
sanaexlAnimeV10_v10 = False # @param  {type:"boolean"}
sanaexlAnimeV10_v11 = False # @param  {type:"boolean"}
SanaeXL_Anime_v1_2_aesthetic = False # @param  {type:"boolean"}
SanaeXL_Anime_v1_3_aesthetic = False # @param  {type:"boolean"}
Illustrious_XL_v0_1 = True # @param  {type:"boolean"}
Illustrious_XL_v0_1_GUIDED = False # @param  {type:"boolean"}
Illustrious_XL_v1_0 = False # @param  {type:"boolean"}
Illustrious_XL_v1_1 = False # @param  {type:"boolean"}
Illustrious_XL_v2_0_stable = False # @param  {type:"boolean"}
jruTheJourneyRemains_v25XL = False # @param  {type:"boolean"}
noobaiXLNAIXL_earlyAccessVersion = False # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred05Version = False # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred075 = False # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred077 = False # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred10Version = False # @param  {type:"boolean"}
noobaiXLNAIXL_epsilonPred11Version = False # @param  {type:"boolean"}
noobaiXLNAIXL_vPredTestVersion = False # @param  {type:"boolean"}
noobaiXLNAIXL_vPred05Version = False # @param  {type:"boolean"}
noobaiXLNAIXL_vPred06Version = False # @param  {type:"boolean"}
noobaiXLNAIXL_vPred065SVersion = False # @param  {type:"boolean"}
noobaiXLNAIXL_vPred075SVersion = False # @param  {type:"boolean"}
noobaiXLNAIXL_vPred09RVersion = False # @param  {type:"boolean"}
noobaiXLNAIXL_vPred10Version = False # @param  {type:"boolean"}
ChenkinNoob_XL_V01 = False # @param  {type:"boolean"}
ponyDiffusionV6XL_v6StartWithThisOne = False # @param  {type:"boolean"}
pdForAnime_v20 = False # @param  {type:"boolean"}
omegaPonyXLAnime_v20 = False # @param  {type:"boolean"}
# @markdown - VAE 模型
vae_ft_ema_560000_ema_pruned = False # @param  {type:"boolean"}
vae_ft_mse_840000_ema_pruned = False # @param  {type:"boolean"}
sdxl_fp16_fix_vae = True # @param  {type:"boolean"}

v1_5_pruned_emaonly and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sd_1.5/v1-5-pruned-emaonly.safetensors", 1])
animefull_final_pruned and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sd_1.5/animefull-final-pruned.safetensors", 1])
sd_xl_base_1_0_0_9vae and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sd_xl_base_1.0_0.9vae.safetensors", 1])
sd_xl_refiner_1_0_0_9vae and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sd_xl_refiner_1.0_0.9vae.safetensors", 1])
sd_xl_turbo_1_0_fp16 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sd_xl_turbo_1.0_fp16.safetensors", 1])
animagine_xl_3_0_base and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-3.0-base.safetensors", 1])
animagine_xl_3_0 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-3.0.safetensors", 1])
animagine_xl_3_1 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-3.1.safetensors", 1])
animagine_xl_4_0 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-4.0.safetensors", 1])
animagine_xl_4_0_opt and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/animagine-xl-4.0-opt.safetensors", 1])
holodayo_xl_2_1 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/holodayo-xl-2.1.safetensors", 1])
kivotos_xl_2_0 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kivotos-xl-2.0.safetensors", 1])
clandestine_xl_1_0 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/clandestine-xl-1.0.safetensors", 1])
UrangDiffusion_1_1 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/UrangDiffusion-1.1.safetensors", 1])
RaeDiffusion_XL_v2 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/RaeDiffusion-XL-v2.safetensors", 1])
kohaku_xl_delta_rev1 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohaku-xl-delta-rev1.safetensors", 1])
kohakuXLEpsilon_rev1 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohakuXLEpsilon_rev1.safetensors", 1])
kohaku_xl_epsilon_rev2 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohaku-xl-epsilon-rev2.safetensors", 1])
kohaku_xl_epsilon_rev3 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohaku-xl-epsilon-rev3.safetensors", 1])
kohaku_xl_zeta and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/kohaku-xl-zeta.safetensors", 1])
starryXLV52_v52 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/starryXLV52_v52.safetensors", 1])
heartOfAppleXL_v20 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/heartOfAppleXL_v20.safetensors", 1])
heartOfAppleXL_v30 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/heartOfAppleXL_v30.safetensors", 1])
sanaexlAnimeV10_v10 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sanaexlAnimeV10_v10.safetensors", 1])
sanaexlAnimeV10_v11 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/sanaexlAnimeV10_v11.safetensors", 1])
SanaeXL_Anime_v1_2_aesthetic and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/SanaeXL-Anime-v1.2-aesthetic.safetensors", 1])
SanaeXL_Anime_v1_3_aesthetic and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/SanaeXL-Anime-v1.3-aesthetic.safetensors", 1])
Illustrious_XL_v0_1 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v0.1.safetensors", 1])
Illustrious_XL_v0_1_GUIDED and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v0.1-GUIDED.safetensors", 1])
Illustrious_XL_v1_0 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v1.0.safetensors", 1])
Illustrious_XL_v1_1 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v1.1.safetensors", 1])
Illustrious_XL_v2_0_stable and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/Illustrious-XL-v2.0-stable.safetensors", 1])
jruTheJourneyRemains_v25XL and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/jruTheJourneyRemains_v25XL.safetensors", 1])
noobaiXLNAIXL_earlyAccessVersion and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_earlyAccessVersion.safetensors", 1])
noobaiXLNAIXL_epsilonPred05Version and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred05Version.safetensors", 1])
noobaiXLNAIXL_epsilonPred075 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred075.safetensors", 1])
noobaiXLNAIXL_epsilonPred077 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred077.safetensors", 1])
noobaiXLNAIXL_epsilonPred10Version and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred10Version.safetensors", 1])
noobaiXLNAIXL_epsilonPred11Version and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_epsilonPred11Version.safetensors", 1])
noobaiXLNAIXL_vPredTestVersion and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPredTestVersion.safetensors", 1])
noobaiXLNAIXL_vPred05Version and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred05Version.safetensors", 1])
noobaiXLNAIXL_vPred06Version and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred06Version.safetensors", 1])
noobaiXLNAIXL_vPred065SVersion and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred065SVersion.safetensors", 1])
noobaiXLNAIXL_vPred075SVersion and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred075SVersion.safetensors", 1])
noobaiXLNAIXL_vPred09RVersion and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred09RVersion.safetensors", 1])
noobaiXLNAIXL_vPred10Version and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/noobaiXLNAIXL_vPred10Version.safetensors", 1])
ChenkinNoob_XL_V01 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/ChenkinNoob-XL-V0.1.safetensors", 1])
ponyDiffusionV6XL_v6StartWithThisOne and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/ponyDiffusionV6XL_v6StartWithThisOne.safetensors", 1])
pdForAnime_v20 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/pdForAnime_v20.safetensors", 1])
omegaPonyXLAnime_v20 and SD_MODEL.append(["https://huggingface.co/licyk/sd-model/resolve/main/sdxl_1.0/omegaPonyXLAnime_v20.safetensors", 1])
vae_ft_ema_560000_ema_pruned and SD_MODEL.append(["https://huggingface.co/licyk/sd-vae/resolve/main/sd_1.5/vae-ft-ema-560000-ema-pruned.safetensors", 1])
vae_ft_mse_840000_ema_pruned and SD_MODEL.append(["https://huggingface.co/licyk/sd-vae/resolve/main/sd_1.5/vae-ft-mse-840000-ema-pruned.safetensors", 1])
sdxl_fp16_fix_vae and SD_MODEL.append(["https://huggingface.co/licyk/sd-vae/resolve/main/sdxl_1.0/sdxl_fp16_fix_vae.safetensors", 1])

##############################################################################
# 下面为初始化参数部分, 不需要修改
INSTALL_PARAMS = {
    "torch_ver":TORCH_VER or None,
    "xformers_ver": XFORMERS_VER or None,
    "git_branch": SD_SCRIPTS_BRANCH or None,
    "git_commit": SD_SCRIPTS_COMMIT or None,
    "model_path": SD_MODEL_PATH or None,
    "model_list": SD_MODEL,
    "use_uv": USE_UV,
    "pypi_index_mirror": PIP_INDEX_MIRROR or None,
    "pypi_extra_index_mirror": PIP_EXTRA_INDEX_MIRROR or None,
    "pypi_find_links_mirror": PIP_FIND_LINKS_MIRROR or None,
    # Kaggle 的环境暂不需要以下镜像源
    # "github_mirror": GITHUB_MIRROR or None,
    # "huggingface_mirror": HUGGINGFACE_MIRROR or None,
    "pytorch_mirror": PYTORCH_MIRROR or None,
    "sd_scripts_repo": SD_SCRIPTS_REPO or None,
    "sd_scripts_requirements": SD_SCRIPTS_REQUIREMENTS or None,
    "retry": RETRY,
    "huggingface_token": HF_TOKEN or None,
    "modelscope_token": MS_TOKEN or None,
    "wandb_token": WANDB_TOKEN or None,
    "git_username": GIT_USER_NAME or None,
    "git_email": GIT_USER_EMAIL or None,
    "check_avaliable_gpu": CHECK_AVALIABLE_GPU,
    "enable_tcmalloc": ENABLE_TCMALLOC,
    "enable_cuda_malloc": ENABLE_CUDA_MALLOC,
    "custom_sys_pkg_cmd": None if sys.platform == "linux" and shutil.which("apt") else [],
    "update_core": UPDATE_CORE,
}
HF_REPO_UPLOADER_PARAMS = {
    "api_type": "huggingface",
    "repo_id": HF_REPO_ID,
    "repo_type": HF_REPO_TYPE,
    "visibility": HF_REPO_VISIBILITY,
    "upload_path": OUTPUT_PATH,
    "retry": RETRY,
}
MS_REPO_UPLOADER_PARAMS = {
    "api_type": "modelscope",
    "repo_id": MS_REPO_ID,
    "repo_type": MS_REPO_TYPE,
    "visibility": MS_REPO_VISIBILITY,
    "upload_path": OUTPUT_PATH,
    "retry": RETRY,
}
os.makedirs(WORKSPACE, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(SD_MODEL_PATH, exist_ok=True)
os.makedirs(INPUT_DATASET_PATH, exist_ok=True)
SD_SCRIPTS_PATH = os.path.join(WORKSPACE, WORKFOLDER)
logger.info("参数设置完成")

In [ ]:
# @title 👇 安装环境
logger.info("开始安装 sd-scripts")
sd_scripts = SDScriptsManager(WORKSPACE, WORKFOLDER)
sd_scripts.install(**INSTALL_PARAMS)
logger.info("sd-scripts 安装完成")

In [ ]:
# @title 👇 模型下载工具: `sd_scripts.get_model()`
# @markdown - 模型下载链接
url = "" # @param {"type":"string","placeholder":"填写模型下载链接"}
# @markdown - 保存的文件名 (可选)
filename = "" # @param {"type":"string","placeholder":"填写模型的名称"}
sd_scripts.get_model(
    url=url,
    path=SD_MODEL_PATH,
    filename=filename if filename else None,
    retry=RETRY,
)

In [ ]:
# @title 👇 模型 / 训练集下载工具: `sd_scripts.repo.download_files_from_repo()`
# @markdown 可从 HuggingFace / ModelScope 仓库下载文件
# @markdown - 仓库类型
api_type = "huggingface" # @param ["huggingface", "modelscope"]
# @markdown - 仓库 ID
repo_id = "" # @param {"type":"string","placeholder":"填写仓库的 ID"}
# @markdown - 仓库类型
repo_type = "model" # @param ["model", "dataset", "space"]
# @markdown 文件在仓库中的路径, 不填写则下载整个仓库
folder = "" # @param {"type":"string","placeholder":"填写文件在仓库中的路径"}
sd_scripts.repo.download_files_from_repo(
    api_type=api_type,
    local_dir=SD_MODEL_PATH,
    repo_id=repo_id,
    repo_type=repo_type,
    folder=folder,
    retry=RETRY,
    num_threads=DOWNLOAD_THREAD,
)

In [ ]:
# @title 👇 压缩包下载工具并解压: `sd_scripts.download_archive_and_unpack()`
# @markdown 支持的压缩包格式为 `ZIP`, `7Z`, `TAR`
# @markdown - 压缩包的下载链接
url = "" # @param {"type":"string","placeholder":"填写压缩包的下载链接"}
# @markdown - 将压缩包进行重命名的名称
name = "" # @param {"type":"string","placeholder":"填写压缩包进行重命名的名称"}

sd_scripts.download_archive_and_unpack(
    url=url,
    local_dir=INPUT_DATASET_PATH,
    name=name if name else None,
    retry=RETRY,
)

In [ ]:
# @title 👇 模型 / 训练集制作工具: `make_dataset()`
# @markdown 基于`sd_scripts.repo.download_files_from_repo()`进行封装</br>
# @markdown 可以自动为下载好的训练集添加重复次数
def make_dataset(
    api_type: str,
    local_dir: str | Path,
    repo_id: str,
    repo_type: str,
    repeat: int,
    folder: str,
) -> None:
    origin_dataset_path = os.path.join(local_dir, folder)
    tmp_dataset_path = os.path.join(local_dir, f"{repeat}_{folder}")
    new_dataset_path = os.path.join(origin_dataset_path, f"{repeat}_{folder}")
    sd_scripts.repo.download_files_from_repo(
        api_type=api_type,
        local_dir=local_dir,
        repo_id=repo_id,
        repo_type=repo_type,
        folder=folder,
        retry=RETRY,
        num_threads=DOWNLOAD_THREAD,
    )
    if os.path.exists(origin_dataset_path):
        logger.info("设置 %s 训练集的重复次数为 %s", folder, repeat)
        shutil.move(origin_dataset_path, tmp_dataset_path)
        shutil.move(tmp_dataset_path, new_dataset_path)
    else:
        logger.error("从 %s 下载 %s 失败", repo_id, folder)
# @markdown - 仓库类型
api_type = "huggingface" # @param ["huggingface", "modelscope"]
# @markdown - 仓库 ID
repo_id = "" # @param {"type":"string","placeholder":"填写仓库 ID"}
# @markdown - 仓库类型
repo_type = "model" # @param ["model", "dataset", "space"]
# @markdown - 训练集文件夹在仓库中的路径
folder = "" # @param {"type":"string","placeholder":"填写训练集文件夹在仓库中的路径"}
# @markdown - 设置训练集的重复次数
repeat = 1 # @param {type:"slider", min:1, max:128, step:1}

make_dataset(
    api_type=api_type,
    local_dir=INPUT_DATASET_PATH,
    repo_id=repo_id,
    repo_type=repo_type,
    repeat=repeat,
    folder=folder,
)

In [ ]:
# @title 👇 模型训练 (可自行修改训练参数)

pretrained_model_name_or_path = "noobaiXLNAIXL_vPred10Version.safetensors" # @param {type:"string"}
vae = "sdxl_fp16_fix_vae.safetensors" # @param {type:"string"}
train_data_dir = "Nachoneko" # @param {type:"string"}
output_name = "Nachoneko_2" # @param {type:"string"}
output_dir = "Nachoneko" # @param {type:"string"}
wandb_run_name = "Nachoneko" # @param {type:"string"}
log_tracker_name = "lora-Nachoneko" # @param {type:"string"}
resolution = "1024,1024" # @param {type:"string"}
save_every_n_epochs = "1" # @param {type:"string"}
max_train_epochs = "2" # @param {type:"string"}
train_batch_size = "6" # @param {type:"string"}
learning_rate = "0.0001" # @param {type:"string"}
unet_lr = "0.0001" # @param {type:"string"}
text_encoder_lr = "0.00001" # @param {type:"string"}
lr_scheduler = "constant_with_warmup" # @param {type:"string"}
lr_warmup_steps = "100" # @param {type:"string"}
optimizer_type = "Lion8bit" # @param {type:"string"}

!python "{SD_SCRIPTS_PATH}/sdxl_train_network.py" \
    --pretrained_model_name_or_path="{SD_MODEL_PATH}/{pretrained_model_name_or_path}" \
    --vae="{SD_MODEL_PATH}/{vae}" \
    --train_data_dir="{INPUT_DATASET_PATH}/{train_data_dir}" \
    --output_name="{output_name}" \
    --output_dir="{OUTPUT_PATH}/{output_dir}" \
    --wandb_run_name="{wandb_run_name}" \
    --log_tracker_name="{log_tracker_name}" \
    --prior_loss_weight=1 \
    --resolution="{resolution}" \
    --enable_bucket \
    --min_bucket_reso=256 \
    --max_bucket_reso=4096 \
    --bucket_reso_steps=64 \
    --save_model_as="safetensors" \
    --save_precision="fp16" \
    --save_every_n_epochs="{save_every_n_epochs}" \
    --max_train_epochs="{max_train_epochs}" \
    --train_batch_size="{train_batch_size}" \
    --gradient_checkpointing \
    --network_train_unet_only \
    --learning_rate="{learning_rate}" \
    --unet_lr="{unet_lr}" \
    --text_encoder_lr="{text_encoder_lr}" \
    --lr_scheduler="{lr_scheduler}" \
    --lr_warmup_steps="{lr_warmup_steps}" \
    --optimizer_type="{optimizer_type}" \
    --network_module="lycoris.kohya" \
    --network_dim=100000 \
    --network_alpha=100000 \
    --network_args \
        conv_dim=100000 \
        conv_alpha=100000 \
        algo=lokr \
        dropout=0 \
        factor=8 \
        train_norm=True \
        preset="full" \
    --optimizer_args \
        weight_decay=0.05 \
        betas="0.9,0.95" \
    --log_with="{LOG_MODULE}" \
    --logging_dir="{OUTPUT_PATH}/logs" \
    --caption_extension=".txt" \
    --shuffle_caption \
    --keep_tokens=0 \
    --max_token_length=225 \
    --seed=1337 \
    --mixed_precision="fp16" \
    --xformers \
    --cache_latents \
    --cache_latents_to_disk \
    --persistent_data_loader_workers \
    --debiased_estimation_loss \
    --vae_batch_size=4 \
    --full_fp16

In [ ]:
# @title 👇 上传模型到 HuggingFace / ModelScope
# 使用 HuggingFace 上传模型
if USE_HF_TO_SAVE_MODEL:
    logger.info("使用 HuggingFace 保存模型")
    sd_scripts.repo.upload_files_to_repo(**HF_REPO_UPLOADER_PARAMS)

# 使用 ModelScope 上传模型
if USE_MS_TO_SAVE_MODEL:
    logger.info("使用 ModelScope 保存模型")
    sd_scripts.repo.upload_files_to_repo(**MS_REPO_UPLOADER_PARAMS)